In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [ ]:
# Task 2: Write your code here:
ppath = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(ppath)

In [ ]:
# Task 3: Write your code here:
df.head()

In [ ]:
# Task 4: Write your code here:
df.info()

In [ ]:
# Task 5: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
df.drop('Order_ID',axis = 1)

In [ ]:
# Task 2: Write your code here:
#see the na and at this moment drop but we will see
df.isna().sum()
df.shape
dff = df.copy()
#df['Wether'].fillna(df['col'].mode()[0])
df = df.dropna()

In [ ]:
# Task 3: Write your code here:
#check dups function
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)


In [ ]:
# Task 4: Write your code here:
#df.info()
#here for the string features
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
def one_hot_encode(y, num_classes):
    y = np.array(y)
    m = len(y)
    # 1. Create a grid of all zeros (num_samples, num_classes)
    one_hot = np.zeros((m, num_classes))

    # 2. Go through each sample one by one
    for i in range(m):
        # Identify which class this sample belongs to
        class_label = int(y[i])

        # In this row (i), set the specific class column to 1
        one_hot[i, class_label] = 1

    return one_hot

  #i wanted to do onehot encoding but the time isnt in my hand
'''wv = df['Weather'].value_counts()
tv = df['Traffic_Level'].value_counts()
TIMv = df['Time_of_Day'].value_counts()
vv = df['Vehicle_Type'].value_counts()
print(wv,tv,TIMv,vv)
one_hot_encode(y, num_classes)
one_hot_encode(y, num_classes)
one_hot_encode(y, num_classes)
one_hot_encode(y, num_classes)'''
#for num in categorical_cols:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for cat in categorical_cols:
  df[cat] = le.fit_transform(df[cat])

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

# Pick only the numerical columns, NOT the target
numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Delivery_Time")

scaler = StandardScaler()

# scale the `numerical_cols`
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

df.head()

In [ ]:
# Task 6: Write your code here:
#for linear reggression the balance isnt a factor of it but outlier is see if its skewed apply a log transform

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df["Delivery_Time"].astype(float)

In [ ]:
mae_e=[]
lr_losses=[]

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error
n_splits = 5  # K=5 Folds
model = RandomForestRegressor()
# 5-Fold shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train & Validate Models
  # Fit the model on train data
  model.fit(X_train, y_train)

  # Use the model to predict the test data
  y_pred = model.predict(X_test)
  loss = y_test-y_pred # it uses the resiual loss true - pred
  mae = mean_absolute_error(y_test,y_pred)
  mae_e.append(mae)
  lr_losses.append(loss)
#lr_losses = lr_losses.value()
#average_losses = np.mean(lr_losses, axis=0)

In [ ]:
# Task 1: Write your code here:

importance_model= list(zip(X.columns, model.feature_importances_))
sorted_importance = sorted(importance_model, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('reandomforest importance Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
# Task 2: Write your code here:


In [ ]:
%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q



In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
models = {

  "Random Forest": RandomForestClassifier(
      n_estimators=200,
      max_depth=10
  ),
  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )
}

In [ ]:
results = {}

for model_name in models:
  results[model_name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': [],'mae':[],}

In [ ]:
# Task Bonus: Write your code here:
for fold_idx, (train_index, test_index) in enumerate(kf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train & Validate Models
  for model_name, model in models.items():

    print(f"Training {model_name}...")

    # Fit the model on train data
    model.fit(X_train, y_train)

    # Use the model to predict the test data
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    mae = mean_absolute_error(y_test,y_pred)
    results[model_name]['accuracy'].append(accuracy)
    results[model_name]['f1'].append(f1)
    results[model_name]['mae'].append(mae)

In [ ]:
for model_name in results:
  print(f"\n{model_name}:")
  # Print the average of each evaluation metric across folds
  print(f"  Accuracy:  {np.mean(results[model_name]['accuracy']):.4f}")
  print(f"  F1-Score:  {np.mean(results[model_name]['f1']):.4f}")
  #print(f"  mae:  {np.mean(results[model_name]['mae']):.4f}")